# Lab 2: API and LLM Integration

Welcome to Lab 2 of Programming Methods! In this class we will be exploring APIs and LLMs. The Lab has **four** main exercises.

After cloning the repository (done in Lab 1) create a new branch for this Lab (e.g., `git checkout -b lab-02-apis`).

Then, complete the following exercises in a **new** Jupyter Notebook. 

Once you are done **commit your work** using the proper Git commands.

Good luck :) 


## A quick note before you start...

APIs let software talk to other software - most apps you use (maps,
weather, payments, social media) are quietly making API calls behind
the scenes. Learning to work with APIs means your code can pull in
live data instead of being limited to what you type by hand.

In this class, the **World Bank API** shows how to fetch open data,
and the **Wikipedia API** shows how to fetch and reuse text/content.
You will then connect both to **Ollama**, a tool for running AI models
*locally*. Once you understand REST APIs, "using AI" is just another API call.

This combination of public data + local AI reflects how a lot of
real software is built: gather data from external sources, then
compute or reason over it.

#### By the end of this class, you should be able to:

- Send GET and POST requests with the `requests` library.
- Pass parameters via query strings, URL paths, and JSON bodies.
- Check HTTP status codes explicitly instead of assuming success.
- Parse nested JSON to extract specific values.
- Explain, at a high level, how API keys/tokens authenticate requests.
- Chain multiple API calls together, using one's output as another's input.

## Setup

Run this cell first. If any of the packages is not installed run `uv add <package-name>`.

In [2]:

import requests
import ollama

## What is a REST API?

A **REST API (Representational State Transfer API)** is a way for applications to communicate with each other over the web using **HTTP**.

A REST API exposes **resources** through URLs called **endpoints**. A client sends an HTTP request to an endpoint, and the server processes the request and returns an HTTP response.

For example, the following Wikipedia endpoint represents information about Portugal:

`https://en.wikipedia.org/api/rest_v1/page/summary/Portugal`

A typical interaction looks like:

**Client → HTTP Request → REST API → HTTP Response → Client**

### HTTP Methods

The HTTP method (or verb) indicates what operation the client wants to perform.

| HTTP Method | Purpose | Example |
|---|---|---|
| **GET** | Retrieve data | Get information about a country |
| **POST** | Send data / create a resource | Send a prompt to an LLM |
| **PUT** | Replace an existing resource | Replace a user profile |
| **PATCH** | Partially update a resource | Update a user's email |
| **DELETE** | Remove a resource | Delete a record |

### Requests

A request can contain information in different places, including:

- **URL path** — identifies the resource.
- **Query parameters** — provide additional options or filters.
- **Headers** — provide information about the request or client.
- **Body** — contains data sent to the server, commonly used with `POST`, `PUT`, and `PATCH`.

For example:

```python
response = requests.get(
    "https://api.worldbank.org/v2/country/pt/indicator/SP.POP.TOTL",
    params={"format": "json", "date": "2022"}
)

---
## Exercise 1: Public REST APIs (World Bank)

1. Choose a country and use the `requests` library to retrieve its **total population** (`SP.POP.TOTL`) from the World Bank API:

   `https://api.worldbank.org/v2/country/{country_code}/indicator/SP.POP.TOTL`

   Pass the following **query parameters** using the `params` argument of `requests.get()`:

   - `format`: request the response in JSON format.

   - `date`: choose the year for which you want to retrieve the population.

2. Explicitly check the HTTP status code returned by the API.

3. If the request is successful:

   - Parse the JSON response.

   - Extract the **country name**, **year**, and **population**.

   - Print these values.

In [6]:
# World Bank API endpoint for Total Population (SP.POP.TOTL) in Portugal (pt)
url = "https://api.worldbank.org/v2/country/pt/indicator/SP.POP.TOTL"

# TODO: no exercício escrever use the params given in the request.get
# Pass the query parameters as a dictionary
params = {
    "format": "json",
    "date": "2022"
}

# Make the GET request
response = requests.get(url, params=params)

# Explicitly check the HTTP status code
if response.status_code == 200:
    data = response.json()
    
    # Parse the nested JSON (World Bank returns [metadata_dict, [data_list]])
    population = data[1][0]['value']
    country = data[1][0]['country']['value']
    year = data[1][0]['date']
    
    print(f"The population of {country} in {year} was {population}.")
else:
    print(f"Failed to retrieve data. HTTP Status Code: {response.status_code}")

The population of Portugal in 2022 was 10434332.


4. Make a second GET request to retrieve the metadata for the population indicator using:

   `https://api.worldbank.org/v2/indicator/SP.POP.TOTL`

   Again, request the response in JSON format using a query parameter.
   

5. If the request is successful:

   - Parse the JSON response.

   - Extract the indicator's **name**, **source**, and **description**.

   - Print these values.

In [7]:
# Endpoint for indicator metadata/documentation
doc_url = "https://api.worldbank.org/v2/indicator/SP.POP.TOTL"

doc_params = {
    "format": "json"
}

doc_response = requests.get(doc_url, params=doc_params)

if doc_response.status_code == 200:
    doc_data = doc_response.json()
    indicator_info = doc_data[1][0]  # same [metadata, [list]] structure as before
    print(doc_data)
    print(f"Indicator: {indicator_info['name']}")
    print(f"Source: {indicator_info['source']['value']}")
    print(f"Description: {indicator_info['sourceNote']}")
else:
    print(f"Failed to retrieve documentation. HTTP Status Code: {doc_response.status_code}")

[{'page': 1, 'pages': 1, 'per_page': '50', 'total': 1}, [{'id': 'SP.POP.TOTL', 'name': 'Population, total', 'unit': '', 'source': {'id': '2', 'value': 'World Development Indicators'}, 'sourceNote': 'Total population is based on the de facto definition of population, which counts all residents regardless of legal status or citizenship. The values shown are midyear estimates.', 'sourceOrganization': 'World Population Prospects, United Nations (UN), uri: https://population.un.org/wpp/, publisher: UN Population Division;\nStatistical databases and publications from national statistical offices, National Statistical Offices (NSOs), uri: https://unstats.un.org/home/nso_sites/, publisher: National Statistical Offices;\nEurostat: Demographic Statistics, Eurostat (ESTAT), uri: https://ec.europa.eu/eurostat/data/database?node_code=earn_ses_monthly, publisher: Eurostat;\nPopulation and Vital Statistics Report (various years), United Nations (UN), uri: https://unstats.un.org, publisher: UN Statist

---
## Exercise 2: Public REST APIs (Wikipedia)

1. Create a list containing the names of **10 countries**.

2. Using a `for` loop, iterate over the countries and use the `requests` library to fetch the summary of each country from the Wikipedia REST API:

   `https://en.wikipedia.org/api/rest_v1/page/summary/{title}`

   Pass the country name as the article title in the URL path.

3. For each request, explicitly check whether the HTTP status code is `200`.

4. If the request is successful:

   - Parse the JSON response.

   - Extract the article's `title`, `extract` (summary), and page URL.

   - Print the extracted information.

5. If the request is not successful, print the country name and the HTTP status code returned by the API.

In [5]:
countries = [
    "Portugal",
    "Spain",
    "France",
    "Germany",
    "Italy",
    "Norway",
    "Japan",
    "Brazil",
    "Canada",
    "Australia",
]

headers = {
    "User-Agent": "NOVA-API-Class/1.0"
}

for country in countries:

    # Wikipedia REST API endpoint for a page summary
    url = f"https://en.wikipedia.org/api/rest_v1/page/summary/{country}"

    response = requests.get(url, headers=headers)

    # Explicitly check the HTTP status code
    if response.status_code == 200:

        # Parse the JSON response
        data = response.json()

        # Extract the relevant information
        page_title = data["title"]
        extract = data["extract"]
        page_url = data["content_urls"]["desktop"]["page"]

        # Print the results
        print(f"Title: {page_title}")
        print(f"Summary: {extract}")
        print(f"URL: {page_url}")
        print("-" * 80)

    else:
        print(
            f"Failed to retrieve {country}. "
            f"HTTP Status Code: {response.status_code}"
        )

Title: Portugal
Summary: Portugal, officially the Portuguese Republic, is a country in Southwestern Europe. Mainland Portugal is located on the southwestern portion of the Iberian Peninsula, and bordered by Spain to the north and east. Portugal also includes the archipelagos of Madeira and the Azores in the Atlantic Ocean. The country has a population of 11.4 million, and Lisbon, its capital, is the largest city. Portugal's internal waters and territorial sea together account for two-fifths of its territory, and its exclusive economic zone is one of Europe's largest. The country's terrain contains a diverse range of landscapes and regional climates.
URL: https://en.wikipedia.org/wiki/Portugal
--------------------------------------------------------------------------------
Title: Spain
Summary: Spain, officially the Kingdom of Spain, is a country in Southern and Western Europe with territories in North Africa. Featuring the southernmost point of continental Europe, it is the largest cou

## Exercise 3: Interacting with Ollama's Local API

Let's connect the World Bank data to our local AI! 

1. Send a **POST** request to Ollama's local API (`http://localhost:11434/api/generate`) using the `requests` library.

2. Create a dictionary payload targeting the `llama3.2:1b` model (or the model you have installed).

3. For the `prompt`, use an f-string to inject the country and population variables you extracted in Exercise 1.

    - *Example: "Act as a grumpy wizard and explain why {country} has exactly {population} people."*

4. Set `stream`: False.

5. Write an if statement to check if the status code is 200. 

6. If the status code is 200, parse the JSON and print only the model's text response.

In [8]:
url = "http://localhost:11434/api/generate"

payload = {
    "model": "llama3.2:1b",
    "prompt": f"Act as a grumpy wizard and explain why {country} has exactly {population} people.",
    "stream": False
}

response = requests.post(url, json=payload)

if response.status_code == 200:
    data = response.json()
    print(data['response'])
else:
    print(f"Failed to retrieve data. HTTP Status Code: {response.status_code}")

(Grumbling) Fine. I'll tell you why Portugal has exactly 104,343,332 people. But don't go thinking I'm some sort of benevolent wizard, guiding you through the intricacies of geography and population counts. No, no. I'm only doing this because I have to, not because I want to.

Now, pay attention, mortal, as I recount the mystical forces that have brought about this astonishing number.

You see, Portugal's population is a result of a complex interplay of historical events, cultural influences, and geographical factors. (Sigh) It's not as if I can simply wave my staff and conjure up the numbers.

Firstly, let's start with the population of Portugal's main islands. The Azores, Madeira, and the mainland (Portugal proper) have populations ranging from around 250,000 to 500,000 people. But, by some sort of wizardly magic, these islands have managed to sustain populations that are roughly 4 times larger than the average European island population. (Muttering to himself) As if they didn't have

7. Send a second request to Ollama: use the Wikipedia extract text from Exercise 2 as the basis for a new prompt (e.g. ask the model to summarize it in one sentence, rewrite it in a different tone, or explain it "like I'm five"). Print the response.

In [10]:

# Trim the extract to keep the prompt a reasonable length
trimmed_extract = extract[:500]

wiki_payload = {
    "model": "llama3.2:1b",
    "prompt": f"Summarize the following text in one sentence: {trimmed_extract}",
    "stream": False
}

wiki_response = requests.post(url, json=wiki_payload)

if wiki_response.status_code == 200:
    wiki_data = wiki_response.json()
    print(wiki_data['response'])
else:
    print(f"Failed to retrieve data. HTTP Status Code: {wiki_response.status_code}")

Australia is a vast country comprising a continent and several smaller islands, boasting a diverse landscape and climate, including deserts and tropical rainforests, due to its large size.


## A Note on Authenticated APIs

The World Bank and Wikipedia APIs don't require
authentication. Many other APIs (e.g. OpenWeatherMap, NASA API, Twitter/X,
GitHub) require an **API key or token** to identify who's making requests,
track usage, and enforce rate limits.

Typically, the key is sent either:
- As a query parameter: `params = {"appid": "YOUR_API_KEY", ...}`
- As a header: `headers = {"Authorization": "Bearer YOUR_TOKEN"}`

You can see an example using NASA's APOD (Astronomy Picture of the Day) API in the cell below.

**⚠️ Keys should never be hardcoded in shared code or notebooks, they're
often loaded from environment variables (`os.environ`) or a `.env` file.**

In [12]:
url = "https://api.nasa.gov/planetary/apod"
params = {"api_key": "DEMO_KEY"}  # rate-limited demo key, no signup needed

response = requests.get(url, params=params)

if response.status_code == 200:
    data = response.json()
    print(data['title'])
    print(data['explanation'])
else:
    print(f"Failed to retrieve data. HTTP Status Code: {response.status_code}")

Apollo 11: Catching Some Sun
Bright sunlight glints as long dark shadows mark this image of the surface of the Moon. It was taken on July 20, 1969, by Apollo 11 astronaut Neil Armstrong, the first to walk on the lunar surface. Pictured is the mission's lunar module, the Eagle, and spacesuited lunar module pilot Buzz Aldrin. Aldrin is unfurling a long sheet of foil also known as the Solar Wind Composition Experiment. Exposed facing the Sun, the foil trapped particles streaming outward in the solar wind, catching a sample of material from the Sun itself. Along with 22 kilograms of moon rocks and lunar soil samples, the solar wind collector was returned for analysis in earthbound laboratories.  APOD's main NASA site is moving: From apod.nasa.gov to science.nasa.gov/apod


## Using the Ollama Python Library

So far, we have interacted with Ollama directly through its HTTP API using `requests`.

Ollama also provides a Python library that simplifies this interaction. Instead of manually creating and sending HTTP requests, we can use functions such as `ollama.chat()`.

First, make sure the library is installed:

`uv add ollama`

In [16]:
response = ollama.chat(
    model="llama3.2:1b",
    messages=[
        {
            "role": "user",
            "content": "What is the capital of Portugal?"
        }
    ]
)

print(response["message"]["content"])

The capital of Portugal is Lisbon.


## Ollama Conversation History

If we want to build a chatbot that can follow a conversation, we need to provide the model with the **conversation history**.

Ollama's `chat()` function accepts a list of messages. Each message contains:

- `role`: who sent the message (`user`, `assistant`, or `system`)
- `content`: the text of the message

For example:

```python
messages = [
    {"role": "user", "content": "What is the capital of Portugal?"},
    {"role": "assistant", "content": "The capital of Portugal is Lisbon."},
    {"role": "user", "content": "What river runs through it?"}
]

For a practical use case check the examples below.

In [14]:
messages = []

user_input = "What is the capital of Portugal?"

messages.append({
    "role": "user",
    "content": user_input
})

response = ollama.chat(
    model="llama3.2:1b",
    messages=messages
)

bot_reply = response["message"]["content"]

messages.append({
    "role": "assistant",
    "content": bot_reply
})

print(bot_reply)

The capital of Portugal is Lisbon.


In [15]:
user_input = "And what is its population?"

messages.append({
    "role": "user",
    "content": user_input
})

response = ollama.chat(
    model="llama3.2:1b",
    messages=messages
)

bot_reply = response["message"]["content"]

messages.append({
    "role": "assistant",
    "content": bot_reply
})

print(bot_reply)

As of my knowledge cutoff in 2023, the estimated population of Lisbon is approximately 505,000 people within the city limits, and around 2.8 million in the metropolitan area. However, please note that population figures may have changed since my knowledge cutoff date.


## Exercise 4: Build a CLI Chatbot

Now that we know how to send messages to Ollama and maintain conversation history, let's build a simple command-line chatbot.

Create a new file called `chatbot.py`.

Your chatbot should:

1. Create an empty `messages` list to store the conversation history.

2. Continuously ask the user for input using a `while` loop.

3. If the user enters `exit` or `quit`, terminate the program.

4. Add each user message to the conversation history using the following format:

   `{"role": "user", "content": user_input}`

5. Send the complete conversation history to `ollama.chat()` using the `llama3.2:1b` model (or the model you have installed).

6. Extract and print the assistant's response.

7. Add the assistant's response to the conversation history so that it is available in the next interaction.

Run your chatbot from the terminal (in the directory where the file is stored):

`python chatbot.py`

Try asking a follow-up question that depends on something you said earlier. Does the chatbot remember the context?

In [ ]:
# TODO: write in a chatbot.py separate script
import ollama
messages = []

print("Chatbot initialized with llama3.2:1b. Type 'exit' to quit.\n")

while True:
    user_input = input("You: ")

    if user_input.lower() in ["exit", "quit"]:
        break

    messages.append({
        "role": "user",
        "content": user_input
    })

    response = ollama.chat(
        model="llama3.2:1b",
        messages=messages
    )

    bot_reply = response["message"]["content"]

    print(f"\nAI: {bot_reply}\n")

    messages.append({
        "role": "assistant",
        "content": bot_reply
    })

Chatbot initialized with llama3.2:1b. Type 'exit' to quit.



## Wrap-up Questions

### 1. Why do we use `GET` with query parameters for the World Bank API, but `POST` with a JSON payload for the Ollama API?

A. Because `GET` only works with public APIs, while `POST` is required for APIs running locally.

B. Because `GET` is typically used to retrieve existing resources, while `POST` can be used to send input to the server for processing.

C. Because query parameters can only be used with `GET`, and JSON can only be used with `POST`.

D. Because `POST` requests are always more secure than `GET` requests.

**Correct answer: B**

---

### 2. Why should we check the HTTP status code before processing the response?

A. To make the API return JSON instead of plain text.

B. To make the request execute faster.

C. To verify that the request succeeded before assuming the response contains the expected data.

D. To prevent the API from receiving too many requests.

**Correct answer: C**

---

### 3. What does `"stream": False` change when making a request to Ollama?

A. It prevents Ollama from accessing the internet.

B. It makes Ollama generate a shorter response.

C. It tells Ollama to return the complete generated response as a single response instead of sending it progressively in chunks.

D. It prevents the model from remembering previous messages.

**Correct answer: C**

---

### 4. Why do we send the complete `messages` history when calling `ollama.chat()`?

A. Because the model does not automatically remember previous API calls, so previous messages must be provided as context.

B. Because Ollama requires at least two messages before it can generate a response.

C. Because storing messages makes the model generate responses faster.

D. Because the `messages` list is automatically stored permanently by Ollama.

**Correct answer: A**

## Optional Stretch Goal: PokeAPI

Follow the same reasoning as in exercise 1 but in this case use the **PokeAPI**.

In [ ]:
import requests

# Students can change this variable to their favorite Pokemon
pokemon_name = "snorlax"
url = f"https://pokeapi.co/api/v2/pokemon/{pokemon_name}"

response = requests.get(url)

if response.status_code == 200:
    data = response.json()
    
    # Extracting standard data
    weight = data['weight']
    
    # Extracting data from a nested list of dictionaries
    types = [t['type']['name'] for t in data['types']]
    
    print(f"{pokemon_name.capitalize()} weighs {weight} hectograms.")
    print(f"Types: {', '.join(types)}")
else:
    print(f"Failed to catch {pokemon_name}! Status Code: {response.status_code}")

## 